<a href="https://colab.research.google.com/github/Ashwini4856/BIS_LAB/blob/main/Genetic_Algorithm_for_Optimization_Problem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam

LR_RANGE = (0.0001, 0.01)
HIDDEN_RANGE = (16, 128)
BATCH_RANGE = (16, 128)

POPULATION_SIZE = 6
GENERATIONS = 5
EPOCHS = 3

def create_chromosome():
    return [
        np.random.uniform(*LR_RANGE),
        np.random.randint(*HIDDEN_RANGE),
        np.random.randint(*BATCH_RANGE)
    ]

def create_model(input_size, classes, learning_rate, hidden_units):
    model = Sequential([
        Dense(hidden_units, activation="relu", input_shape=(input_size,)),
        Dense(classes, activation="softmax")
    ])

    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

def fitness(chromosome, X_train, y_train, X_val, y_val):
    learning_rate = chromosome[0]
    hidden_units = int(round(chromosome[1]))
    batch_size = int(round(chromosome[2]))

    model = create_model(
        X_train.shape[1],
        len(np.unique(y_train)),
        learning_rate,
        hidden_units
    )

    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=batch_size,
        verbose=0
    )

    return max(history.history["val_accuracy"])

def crossover(parent1, parent2):
    alpha = np.random.random()

    return [
        alpha * parent1[i] + (1 - alpha) * parent2[i]
        for i in range(3)
    ]

def mutation(chromosome):
    child = chromosome.copy()

    child[0] += np.random.normal(0, 0.001)
    child[1] += np.random.normal(0, 10)
    child[2] += np.random.normal(0, 10)

    child[0] = np.clip(child[0], *LR_RANGE)
    child[1] = np.clip(child[1], *HIDDEN_RANGE)
    child[2] = np.clip(child[2], *BATCH_RANGE)

    return child

def genetic_algorithm(X_train, y_train, X_val, y_val):
    population = [
        create_chromosome()
        for _ in range(POPULATION_SIZE)
    ]

    best_chromosome = None
    best_accuracy = 0

    for generation in range(GENERATIONS):
        results = []

        for chromosome in population:
            accuracy = fitness(
                chromosome,
                X_train,
                y_train,
                X_val,
                y_val
            )

            results.append((chromosome, accuracy))

            if accuracy > best_accuracy:
                best_accuracy = accuracy
                best_chromosome = chromosome.copy()

        results.sort(key=lambda x: x[1], reverse=True)

        parents = [
            x[0] for x in results[:POPULATION_SIZE // 2]
        ]

        new_population = parents.copy()

        while len(new_population) < POPULATION_SIZE:
            i, j = np.random.choice(
                len(parents), 2, replace=False
            )

            child = crossover(parents[i], parents[j])
            child = mutation(child)

            new_population.append(child)

        population = new_population

        print(
            "Generation:", generation + 1,
            "Best Accuracy:", round(best_accuracy, 4)
        )

    return best_chromosome, best_accuracy

(X_train, y_train), (X_test, y_test) = tf.keras.datasets.mnist.load_data()

X_train = X_train.reshape(-1, 784) / 255.0
X_test = X_test.reshape(-1, 784) / 255.0

X_val = X_train[-10000:]
y_val = y_train[-10000:]

X_train = X_train[:-10000]
y_train = y_train[:-10000]

best_params, best_accuracy = genetic_algorithm(
    X_train,
    y_train,
    X_val,
    y_val
)

print("\nBest Hyperparameters")
print("Learning Rate:", best_params[0])
print("Hidden Units:", int(round(best_params[1])))
print("Batch Size:", int(round(best_params[2])))
print("Validation Accuracy:", best_accuracy)